# Experiment 2 multivariate analysis

This reruns the historical block sequence and robustness checks. The main model assignments reproduce the documented retained blocks; they are not a new automatic selection procedure. The missing H-prime efficiency assignment has been restored. Historical narrative decisions are descriptions of the original sample and may not hold for fresh simulations. Review convergence and diagnostics before interpreting any new fit.

In [ ]:
from pathlib import Path
for folder in ['raw_data', 'results/main-analysis']:
    Path(folder).mkdir(parents=True, exist_ok=True)


# Multivariate Analysis

This notebook contains the main factorial multivariate analyses on efficiency and accuracy.

In [ ]:
import pandas as pd
from polygraphs.analysis import Processor
import networkx as nx
import statsmodels.formula.api as smf
import statsmodels.api as sm
from patsy import dmatrix
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100

## Preparation - Accuracy Analysis

In [ ]:
df_accuracy = pd.read_csv("raw_data/main_factorial.csv")

In [ ]:
df_accuracy.head()

In [ ]:
df_accuracy.columns

### Encoding Outcome

1 = simulation reaches correct consensus on B within 100,000 steps

0 = simulation does not do so within 100,000 steps

In [ ]:
df_accuracy['outcome'] = ((df_accuracy['action'] == 'B') & (df_accuracy['steps'] <= 100000)).astype(int)

In [ ]:
df_accuracy.head()

## Preparation - Efficiency Analysis

In [ ]:
df_efficiency = pd.read_csv("raw_data/main_factorial.csv")

In [ ]:
df_efficiency.head()

### Encoding Outcome

In [ ]:
df_efficiency = df_efficiency[df_efficiency['action'] == 'B']
df_efficiency = df_efficiency.rename(columns={'steps': 'steps_to_B'})

In [ ]:
df_efficiency.shape

In [ ]:
df_efficiency['action'].value_counts()

In [ ]:
df_efficiency.head()

### Checking for Overdispersion

In [ ]:
formula = 'steps_to_B ~ C(environment) + C(strategy) + C(network_kind)'

In [ ]:
#fitting models and calculating disspersion
pois = smf.glm(formula, data=df_efficiency, family=sm.families.Poisson()).fit()
dispersion = pois.pearson_chi2 / pois.df_resid
negative_bin = smf.negativebinomial(formula, data=df_efficiency).fit(disp=False)

In [ ]:
print(f'Pearson dispersion:  {dispersion:,.2f}')
print(f'Poisson AIC:         {pois.aic:,.1f}')
print(f'NB AIC:              {negative_bin.aic:,.1f}')
print(f'Estimated alpha:     {negative_bin.params["alpha"]:.3f}')

Since dispersion is much greater than 1.5 the suitable model to use is **negative binomial**.

## Predictors - First Order Independent Variables

Seven first order independent predictors will be included, three categorical and four continitious:.

#### Categorical:
- Environment - misinformation vs disinformation
- Strategy - gullible strategy vs aligned strategy
- Network kind - Erdos-Renyi, Barabasi-Albert and Wattz Strogatz

#### Continious:
- Density
- Average Local Clustering
- Average Local Path length
- Reliable nodes count

Continious variables are not currently present in either the accuracy nor the efficiency dataframes, in the following cells the StructureProcessor is used to enrich the dataset.

In [ ]:
class StructureProcessor(Processor):

    def __init__(self, path, include=None, exclude=None, config_check=False):
        super().__init__(path, include, exclude, config_check)
        self.density()
        self.reliable_node_count()
        self.clustering()
        self.path_length()

    def density(self):
        self.dataframe['density'] = [nx.density(g) for g in self.graphs]

    def reliable_node_count(self):
        self.dataframe['reliable_node_count'] = [int(g.pg['ndata']['reliability'].sum().item()) for g in self.graphs]

    def clustering(self):
        self.dataframe['clustering'] = [nx.average_clustering(g) for g in self.graphs]

    def path_length(self):
        out = []
        for g in self.graphs:
            if nx.is_connected(g):
                out.append(nx.average_shortest_path_length(g))
            else:
                largest = max(nx.connected_components(g), key=len)
                out.append(nx.average_shortest_path_length(g.subgraph(largest)))
        self.dataframe['path_length'] = out

In [ ]:
data_path = "../../outputs/experiment_2" 

In [ ]:
unreliable_ops = [
    'UnreliableNetworkBasicGullibleBinomialOp',
    'UnreliableNetworkModifiedAlignedBinomialOp',
    'UnreliableNetworkBasicGullibleNegativeEpsOp',
    'UnreliableNetworkModifiedAlignedNegativeEpsOp',
]

In [ ]:
frames = []
for kind in ['random', 'wattsstrogatz', 'barabasialbert']:
    for op in unreliable_ops:
        sp = StructureProcessor(
            data_path,
            include={
                'network.kind': kind,
                'network.size': 64,
                'epsilon': 0.001,
                'reliability': 0.75,
                'op': op,},)
        sp.add_config()
        frames.append(sp.get().copy())

In [ ]:
df_complete = pd.concat(frames, ignore_index=True)
df_complete.to_csv('raw_data/main_factorial_complete.csv', index=False)
print(df_complete.shape)

Now we redefine the accuracy and efficiency datasets with the same encodings defined previously

In [ ]:
#Use this if you have access to the csv on this or another path and you don't want to wait so long!!
df_complete = pd.read_csv('raw_data/main_factorial_complete.csv')

In [ ]:
df_accuracy = df_complete.copy()
df_accuracy['outcome'] = ((df_accuracy['action'] == 'B') & (df_accuracy['steps'] <= 100000)).astype(int)

In [ ]:
df_accuracy.head()

In [ ]:
df_efficiency = df_complete[df_complete['action'] == 'B'].copy()
df_efficiency = df_efficiency.rename(columns={'steps': 'steps_to_B'})

In [ ]:
df_efficiency.head()

#### Continious Variable Standardisation

$$
z_i = \frac{x_i - \bar{x}}{s_x}
$$

In [ ]:
#continiouts predictors
continious_pred = ['density', 'clustering', 'path_length', 'reliable_node_count']

In [ ]:
#standardise
df_accuracy[[c + '_z' for c in continious_pred]] = (df_accuracy[continious_pred] - df_accuracy[continious_pred].mean()) / df_accuracy[continious_pred].std()
df_efficiency[[c + '_z' for c in continious_pred]] = (df_efficiency[continious_pred] - df_accuracy[continious_pred].mean()) / df_accuracy[continious_pred].std()

In [ ]:
df_accuracy[[c + '_z' for c in continious_pred]].agg(['mean', 'std']).round(6)

#### Colinearity Check

In [ ]:
continious_z = ['density_z', 'clustering_z', 'path_length_z', 'reliable_node_count_z']

In [ ]:
corr = df_accuracy[continious_z].corr()
corr.round(3)

FLAG: **density × clustering**

#### Variance Inflation Factor (VIF)

In [ ]:
X = df_accuracy[continious_z].values
vif = pd.DataFrame({'predictor': continious_z,'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])],})
vif.round(2)

Conclusion: No predictors need to be dropped.

## First Order Interaction Effects 

### Fitting first order effects models (accuracy and efficiency separately)

#### Accuracy Baseline - Logistic Regression

In [ ]:
# The base formula from the plan
base_formula = 'outcome ~ C(environment) + C(strategy) + C(network_kind) + density_z + clustering_z + path_length_z + reliable_node_count_z'

In [ ]:
def env_from_op(op):
    if 'NegativeEps' in op:
        return 'disinfo'
    elif 'Binomial' in op:
        return 'misinfo'
    return None

def strat_from_op(op):
    if 'Gullible' in op:
        return 'gullible'
    elif 'ModifiedAligned' in op:
        return 'aligned'
    return None

In [ ]:
df_accuracy['environment'] = df_accuracy['op'].map(env_from_op)
df_accuracy['strategy'] = df_accuracy['op'].map(strat_from_op)

In [ ]:
df_efficiency['environment'] = df_efficiency['op'].map(env_from_op)
df_efficiency['strategy'] = df_efficiency['op'].map(strat_from_op)

In [ ]:
df_accuracy.groupby(['environment', 'strategy']).size()

In [ ]:
accuracy_baseline = smf.logit(base_formula, data=df_accuracy).fit(disp=False)

In [ ]:
print('ACCURACY (logistic)')
print(f'Converged:{accuracy_baseline.mle_retvals["converged"]}')
print(f'AIC: {accuracy_baseline.aic:,.1f}')
print(f'BIC: {accuracy_baseline.bic:,.1f}')
print(f'Pseudo R²: {accuracy_baseline.prsquared:.4f}')
print(f'Log-likelihood: {accuracy_baseline.llf:,.1f}')
print(f'Largest |coef|: {accuracy_baseline.params.abs().max():.3f}')
print(f'Largest SE: {accuracy_baseline.bse.max():.3f}')
print()
print(accuracy_baseline.summary())

#### Efficiency Baseline - Negative Binomial

In [ ]:
efficiency_base_formula = 'steps_to_B ~ C(environment) + C(strategy) + C(network_kind) + density_z + clustering_z + path_length_z + reliable_node_count_z'

In [ ]:
efficiency_baseline = smf.negativebinomial(efficiency_base_formula, data=df_efficiency).fit(disp=False)

In [ ]:
print('EFFICIENCY (negative binomial)')
print(f'Converged: {efficiency_baseline.mle_retvals["converged"]}')
print(f'Iterations: {efficiency_baseline.mle_retvals.get("iterations", "n/a")}')
print(f'AIC: {efficiency_baseline.aic:,.1f}')
print(f'BIC: {efficiency_baseline.bic:,.1f}')
print(f'Log-likelihood: {efficiency_baseline.llf:,.1f}')
print(f'Largest |coef|: {efficiency_baseline.params.abs().max():.3f}')
print(f'Largest SE: {efficiency_baseline.bse.max():.3f}')
print(f'Estimated alpha: {efficiency_baseline.params["alpha"]:.3f}')
print()
print(efficiency_baseline.summary())

#### Saving baseline models

In [ ]:
accuracy_current = accuracy_baseline
efficiency_current = efficiency_baseline

### Iterative Block Addition

For the iterative block addition we proceed with the following pattern:
1. Current Model = First order model
2. New model = First order model + Block
3. If pass criteria Current Model = New Model
4. Otherwise: Keep Current Model as is.
5. Test successively for each block A-E above.

In addition, the following function is defined to make the comparison faster:

In [ ]:
def block_test(current, new, name=''):
    """
    A function to automatically compare the new model with the current model.
    """
    LR = 2 * (new.llf - current.llf)
    df_diff = new.df_model - current.df_model
    p_value = chi2.sf(LR, df_diff)
    delta_aic = current.aic - new.aic
    delta_bic = current.bic - new.bic
    converged = new.mle_retvals.get('converged', True)
    
    #implementing of plan's rule of thumb
    keep = (p_value < 0.05 and delta_aic >= 2 and delta_bic > -2 and converged)
    
    print(f'{name}:')
    print(f'LR statistic: {LR:.2f}  (df = {df_diff})')
    print(f'LR p-value: {p_value:.4g}')
    print(f'ΔAIC: {delta_aic:+.2f}({"better" if delta_aic > 0 else "worse"})')
    print(f'ΔBIC: {delta_bic:+.2f}({"better" if delta_bic > 0 else "worse"})')
    print(f'Converged: {converged}')
    print(f'Decision: {"KEEP" if keep else "DROP"}')
    return keep

#### Block A

- $\text{Environment} \times \text{Strategy}$

**Formulas**

In [ ]:
block_a_accuracy_formula = base_formula + ' + C(environment):C(strategy)'
block_a_efficiency_formula = block_a_accuracy_formula.replace('outcome', 'steps_to_B')

**Fitting Models**

In [ ]:
accuracy_a = smf.logit(block_a_accuracy_formula, data=df_accuracy).fit(disp=False)
efficiency_a = smf.negativebinomial(block_a_efficiency_formula, data=df_efficiency).fit(disp=False)

In [ ]:
print('BLOCK A: environment × strategy\n')
keep_a_acc = block_test(accuracy_current, accuracy_a, 'accuracy')
print()
keep_a_eff = block_test(efficiency_current, efficiency_a, 'efficiency')

**Decision**

$$
\text{\textbf{DROP} accuracy (block A) and \textbf{KEEP} efficiency (block A)}
$$

In [ ]:
efficiency_current = efficiency_a

#### Block B

- $\text{Strategy} \times \text{Network Kind}$

**Formulas**

In [ ]:
accuracy_formula = accuracy_current.model.formula
efficiency_formula = efficiency_current.model.formula
block_b_accuracy_formula = accuracy_formula + ' + C(strategy):C(network_kind)'
block_b_efficiency_formula = efficiency_formula + ' + C(strategy):C(network_kind)'

**Fitting Models**

In [ ]:
accuracy_b = smf.logit(block_b_accuracy_formula, data=df_accuracy).fit(disp=False)

In [ ]:
efficiency_b = smf.negativebinomial(block_b_efficiency_formula, data=df_efficiency).fit(disp=False)

**Comparison Test**

In [ ]:
print('BLOCK B: strategy × network_kind\n')
keep_b_acc = block_test(accuracy_current, accuracy_b, 'accuracy')
print()
keep_b_eff = block_test(efficiency_current, efficiency_b, 'efficiency')

**Decision**

$$
\text{No model gets updated}
$$

#### Block C

- $\text{Environment} \times \text{Density}$
- $\text{Environment} \times \text{Clustering}$
- $\text{Environment} \times \text{Path Length}$
- $\text{Environment} \times \text{Reliable Node Count}$

**Formulas**

In [ ]:
#getting the current formulas
accuracy_formula = accuracy_current.model.formula
efficiency_formula = efficiency_current.model.formula

In [ ]:
#updating the formulas for block c
block_c_accuracy_formula = accuracy_formula + ' + C(environment):density_z' + ' + C(environment):clustering_z' + ' + C(environment):path_length_z' + ' + C(environment):reliable_node_count_z'
block_c_efficiency_formula = efficiency_formula + ' + C(environment):density_z'+ ' + C(environment):clustering_z' + ' + C(environment):path_length_z' + ' + C(environment):reliable_node_count_z'

**Fitting Models**

In [ ]:
accuracy_c = smf.logit(block_c_accuracy_formula, data=df_accuracy).fit(disp=False)

In [ ]:
efficiency_c = smf.negativebinomial(block_c_efficiency_formula, data=df_efficiency).fit(disp=False)

**NB Non Convergence: Block C - individual term tests on the current efficiency model**

In [ ]:
base_aic = efficiency_current.aic
base_formula = efficiency_current.model.formula

In [ ]:
block_c_terms = ['C(environment):density_z','C(environment):clustering_z','C(environment):path_length_z','C(environment):reliable_node_count_z']

In [ ]:
for term in block_c_terms:
    fit = smf.negativebinomial(base_formula + ' + ' + term, data=df_efficiency).fit(disp=False)
    converged = fit.mle_retvals.get('converged', False)
    delta_aic = base_aic - fit.aic
    matching = [p for p in fit.params.index if 'environment' in p and term.split(':')[1] in p]
    print(f'{term}')
    print(f'  Converged: {converged}    ΔAIC: {delta_aic:+.2f}')
    for p in matching:
        print(f'  {p}: coef = {fit.params[p]:.3f}, SE = {fit.bse[p]:.3f}')
    if converged == True and delta_aic >= 6:
        print("FLAG: CONVERGED AND DELTA AIC IS GREAT THAN OR EQUAL TO 6")
    if converged == False:
        print("FLAG: DID NOT CONVERGE!!!")
    print()

**Colinearity Diagnostic (NB C)**

In [ ]:
def collinearity_diagnostic(model, name=''):
    """
    This function takes a fitted model, then uses its design matrix to return the VIFs and top pairwise correlations.
    """
    exog = model.model.exog
    names = model.model.exog_names[:exog.shape[1]]
    X = pd.DataFrame(exog, columns=names)
    X = X.drop(columns=['Intercept']) #dropping the intercept
    
    #For every column we get the VIF
    vif = pd.DataFrame({'predictor': X.columns, 'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]}).sort_values('VIF', ascending=False).reset_index(drop=True)
    
    #Then we get the top pairwise correlations
    corr = X.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    pairs = corr.where(mask).stack().reset_index()
    pairs.columns = ['var1', 'var2', 'r']
    pairs = pairs.reindex(pairs['r'].abs().sort_values(ascending=False).index).head(10)

    #and print the result
    print(f'{name}: Collinearity diagnostic')
    print(f'Shape: {X.shape}\n')
    print('VIF:')
    print(vif.round(2).to_string(index=False))
    print()
    print('Top 10 absolute pairwise correlations')
    print(pairs.round(3).to_string(index=False))
    print()
    return vif, corr

In [ ]:
vif_c, corr_c = collinearity_diagnostic(efficiency_c, 'BLOCK C')

**Comparison Test**

In [ ]:
print('BLOCK C: environment x structural predictors\n')
keep_c_acc = block_test(accuracy_current, accuracy_c, 'accuracy')
print()
keep_c_eff = block_test(efficiency_current, efficiency_c, 'efficiency')

**Decision**

$$
\text{\textbf{KEEP} accuracy (block C) and \textbf{DROP} efficiency (block C)}
$$

In [ ]:
accuracy_current = accuracy_c

#### Block D

- $\text{Network Kind} \times \text{Density}$
- $\text{Network Kind} \times \text{Clustering}$
- $\text{Network Kind} \times \text{Path Length}$
- $\text{Network Kind} \times \text{Reliable Node Count}$

**Formulas**

In [ ]:
#current formulas
accuracy_formula = accuracy_current.model.formula
efficiency_formula = efficiency_current.model.formula

In [ ]:
# Block d formilas
block_d_accuracy_formula = accuracy_formula + ' + C(network_kind):density_z' + ' + C(network_kind):clustering_z' + ' + C(network_kind):path_length_z' + ' + C(network_kind):reliable_node_count_z'
block_d_efficiency_formula = efficiency_formula + ' + C(network_kind):density_z' + ' + C(network_kind):clustering_z' + ' + C(network_kind):path_length_z' + ' + C(network_kind):reliable_node_count_z'

**Fitting Models**

In [ ]:
accuracy_d = smf.logit(block_d_accuracy_formula, data=df_accuracy).fit(disp=False)

In [ ]:
efficiency_d = smf.negativebinomial(block_d_efficiency_formula, data=df_efficiency).fit(disp=False)

**Colinearity Diagnostic (NB D)**

In [ ]:
vif_d, corr_d = collinearity_diagnostic(efficiency_d, 'BLOCK D')

**NB Non Convergence: Block D individual term tests on the current efficiency model**

In [ ]:
base_aic = efficiency_current.aic
base_formula = efficiency_current.model.formula

In [ ]:
block_d_terms = [
    'C(network_kind):density_z',
    'C(network_kind):clustering_z',
    'C(network_kind):path_length_z',
    'C(network_kind):reliable_node_count_z',
]

In [ ]:
for term in block_d_terms:
    fit = smf.negativebinomial(base_formula + ' + ' + term, data=df_efficiency).fit(disp=False)
    converged = fit.mle_retvals.get('converged', False)
    delta_aic = base_aic - fit.aic
    matching = [p for p in fit.params.index if 'network_kind' in p and term.split(':')[1] in p]
    print(f'{term}')
    print(f'  Converged: {converged}    ΔAIC: {delta_aic:+.2f}')
    for p in matching:
        print(f'  {p}: coef = {fit.params[p]:.3f}, SE = {fit.bse[p]:.3f}')
    if converged == True and delta_aic >= 6:
        print("FLAG: CONVERGED AND DELTA AIC IS GREAT THAN OR EQUAL TO 6")
    if converged == False:
        print("FLAG: DID NOT CONVERGE!!!")
    print()

**Comparison Test**

In [ ]:
print('BLOCK D: network_kind × structural predictors\n')
keep_d_acc = block_test(accuracy_current, accuracy_d, 'accuracy')
print()
keep_d_eff = block_test(efficiency_current, efficiency_d, 'efficiency')

**Decision**

$$
\text{No model gets updated}
$$

#### Block E

- $\text{Density} \times \text{Clustering}$
- $\text{Density} \times \text{Path Length}$
- $\text{Path Length} \times \text{Clustering}$

**Formulas**

In [ ]:
#Current
accuracy_formula = accuracy_current.model.formula
efficiency_formula = efficiency_current.model.formula

In [ ]:
#formuilas for block E
block_e_accuracy_formula = accuracy_formula+ ' + density_z:clustering_z' + ' + density_z:path_length_z' + ' + path_length_z:clustering_z'
block_e_efficiency_formula = efficiency_formula + ' + density_z:clustering_z' + ' + density_z:path_length_z' + ' + path_length_z:clustering_z'

**Fit Models**

In [ ]:
accuracy_e = smf.logit(block_e_accuracy_formula, data=df_accuracy).fit(disp=False)

In [ ]:
efficiency_e = smf.negativebinomial(block_e_efficiency_formula, data=df_efficiency).fit(disp=False)

**Comparison Test**

In [ ]:
print('BLOCK E: structural × structural\n')
keep_e_acc = block_test(accuracy_current, accuracy_e, 'accuracy')
print()
keep_e_eff = block_test(efficiency_current, efficiency_e, 'efficiency')

**Decision**

$$
\text{\textbf{KEEP} accuracy (block E) and \textbf{KEEP} efficiency (block E)}
$$

In [ ]:
accuracy_current = accuracy_e
efficiency_current = efficiency_e

#### Block F

- $\text{Reliable Node Count} \times \text{Density}$
- $\text{Reliable Node Count} \times \text{Clustering}$
- $\text{Reliable Node Count} \times \text{Path Length}$

**Formulas**

In [ ]:
#current
accuracy_formula = accuracy_current.model.formula
efficiency_formula = efficiency_current.model.formula

In [ ]:
block_f_accuracy_formula = accuracy_formula+ ' + reliable_node_count_z:density_z' + ' + reliable_node_count_z:clustering_z' + ' + reliable_node_count_z:path_length_z'
block_f_efficiency_formula = efficiency_formula + ' + reliable_node_count_z:density_z' + ' + reliable_node_count_z:clustering_z' + ' + reliable_node_count_z:path_length_z'

**Fitting Models**

In [ ]:
accuracy_f = smf.logit(block_f_accuracy_formula, data=df_accuracy).fit(disp=False)

In [ ]:
efficiency_f = smf.negativebinomial(block_f_efficiency_formula, data=df_efficiency).fit(disp=False)

**NB Non Convergence: Block F - individual term tests on the current efficiency model**

In [ ]:
base_aic = efficiency_current.aic
base_formula = efficiency_current.model.formula

In [ ]:
block_f_terms = ['reliable_node_count_z:density_z','reliable_node_count_z:clustering_z', 'reliable_node_count_z:path_length_z']

In [ ]:
for term in block_f_terms:
    fit = smf.negativebinomial(base_formula + ' + ' + term, data=df_efficiency).fit(disp=False)
    converged = fit.mle_retvals.get('converged', False)
    delta_aic = base_aic - fit.aic
    if term in fit.params.index:
        coef = fit.params[term]
        se = fit.bse[term]
    else:
        coef, se = None, None
    print(f'{term}')
    print(f'  Converged: {converged}    ΔAIC: {delta_aic:+.2f}')
    if coef is not None:
        print(f'  coef = {coef:.3f}, SE = {se:.3f}')
    if converged == True and delta_aic >= 6:
        print("FLAG: CONVERGED AND DELTA AIC IS GREAT THAN OR EQUAL TO 6")
    if converged == False:
        print("FLAG: DID NOT CONVERGE!!!")
    print()

**Colinearity Diagnostic (NB F)**

In [ ]:
vif_f, corr_f = collinearity_diagnostic(efficiency_f, 'BLOCK F')

**Comparison Test**

In [ ]:
print('BLOCK F: reliable_node_count × structural\n')
keep_f_acc = block_test(accuracy_current, accuracy_f, 'accuracy')
print()
keep_f_eff = block_test(efficiency_current, efficiency_f, 'efficiency')

**Decision**

$$
\text{No model gets updated}
$$

#### Block G

- $\text{Strategy} \times \text{Density}$
- $\text{Strategy} \times \text{Clustering}$
- $\text{Strategy} \times \text{Path Length}$
- $\text{Strategy} \times \text{Reliable Node Count}$

**Formulas**

In [ ]:
accuracy_formula = accuracy_current.model.formula
efficiency_formula = efficiency_current.model.formula

In [ ]:
block_g_accuracy_formula = accuracy_formula+ ' + C(strategy):density_z' + ' + C(strategy):clustering_z' + ' + C(strategy):path_length_z' + ' + C(strategy):reliable_node_count_z'
block_g_efficiency_formula = efficiency_formula + ' + C(strategy):density_z' + ' + C(strategy):clustering_z' + ' + C(strategy):path_length_z' + ' + C(strategy):reliable_node_count_z'

**Fitting models**

In [ ]:
accuracy_g = smf.logit(block_g_accuracy_formula, data=df_accuracy).fit(disp=False)

In [ ]:
efficiency_g = smf.negativebinomial(block_g_efficiency_formula, data=df_efficiency).fit(disp=False)

**NB Non Convergence: Block G - individual term tests on the current efficiency model**

In [ ]:
base_aic = efficiency_current.aic
base_formula = efficiency_current.model.formula

In [ ]:
block_g_terms = ['C(strategy):density_z','C(strategy):clustering_z','C(strategy):path_length_z', 'C(strategy):reliable_node_count_z']

In [ ]:
for term in block_g_terms:
    fit = smf.negativebinomial(base_formula + ' + ' + term, data=df_efficiency).fit(disp=False)
    converged = fit.mle_retvals.get('converged', False)
    delta_aic = base_aic - fit.aic
    matching = [p for p in fit.params.index if 'strategy' in p and term.split(':')[1] in p]
    print(f'{term}')
    print(f'  Converged: {converged}    ΔAIC: {delta_aic:+.2f}')
    for p in matching:
        print(f'  {p}: coef = {fit.params[p]:.3f}, SE = {fit.bse[p]:.3f}')
    if converged == True and delta_aic >= 6:
        print("FLAG: CONVERGED AND DELTA AIC IS GREAT THAN OR EQUAL TO 6")
    if converged == False:
        print("FLAG: DID NOT CONVERGE!!!")
    print()

**Colinearity Diagnostic (NB G)**

In [ ]:
vif_g, corr_g = collinearity_diagnostic(efficiency_g, 'BLOCK G')

**Comparison**

In [ ]:
print('BLOCK G: strategy × structural predictors\n')
keep_g_acc = block_test(accuracy_current, accuracy_g, 'accuracy')
print()
keep_g_eff = block_test(efficiency_current, efficiency_g, 'efficiency')

**Decision**

$$
\text{\textbf{KEEP} accuracy (block G) and \textbf{DROP} efficiency (block G)}
$$

In [ ]:
accuracy_current = accuracy_g

### Block H

Following the bservation that the reliable_node_count interactions were among the cleanest terms (lowest VIFs and lowest pairwise correlations) within their original blocks but were discarded along with their parent blocks (C, D, F, G), Block H tests the remaining RNC interactions on their own merits.

For accuracy, `C(environment):reliable_node_count_z` and `C(strategy):reliable_node_count_z` are already retained via Blocks C and G; Block H adds the remaining four. For efficiency, no RNC interactions survived A–G; Block H adds all six.

In [ ]:
accuracy_formula = accuracy_current.model.formula
efficiency_formula = efficiency_current.model.formula

In [ ]:
block_h_accuracy_formula = accuracy_formula + " + C(network_kind):reliable_node_count_z" + " + reliable_node_count_z:density_z" + " + reliable_node_count_z:clustering_z" + " + reliable_node_count_z:path_length_z"
block_h_efficiency_formula = efficiency_formula + " + C(environment):reliable_node_count_z" + " + C(strategy):reliable_node_count_z" + " + C(network_kind):reliable_node_count_z" + " + reliable_node_count_z:density_z"+ " + reliable_node_count_z:clustering_z" + " + reliable_node_count_z:path_length_z"

In [ ]:
accuracy_h = smf.logit(block_h_accuracy_formula, data=df_accuracy).fit(disp=False)

In [ ]:
efficiency_h = smf.negativebinomial(block_h_efficiency_formula, data=df_efficiency).fit(disp=False)

**NB Non-Convergence: Block H individual term tests on the current efficiency model**

In [ ]:
block_h_terms = [
    "C(environment):reliable_node_count_z",
    "C(strategy):reliable_node_count_z",
    "C(network_kind):reliable_node_count_z",
    "reliable_node_count_z:density_z",
    "reliable_node_count_z:clustering_z",
    "reliable_node_count_z:path_length_z",
]

In [ ]:
for term in block_h_terms:
    fit = smf.negativebinomial(base_formula + ' + ' + term, data=df_efficiency).fit(disp=False)
    converged = fit.mle_retvals.get('converged', False)
    delta_aic = base_aic - fit.aic
    matching = [p for p in fit.params.index if 'strategy' in p and term.split(':')[1] in p]
    print(f'{term}')
    print(f'  Converged: {converged}    ΔAIC: {delta_aic:+.2f}')
    for p in matching:
        print(f'  {p}: coef = {fit.params[p]:.3f}, SE = {fit.bse[p]:.3f}')
    if converged == True and delta_aic >= 6:
        print("FLAG: CONVERGED AND DELTA AIC IS GREAT THAN OR EQUAL TO 6")
    if converged == False:
        print("FLAG: DID NOT CONVERGE!!!")
    print()

In [ ]:
vif_h, corr_h = collinearity_diagnostic(efficiency_h, 'BLOCK H')

**Comparison**

In [ ]:
print('BLOCK H: reliable_node_count interactions\n')
keep_h_acc = block_test(accuracy_current, accuracy_h, 'accuracy')
print()
keep_h_eff = block_test(efficiency_current, efficiency_h, 'efficiency')

### Block H′

#### Rationale

Block H tested all remaining reliable_node_count interactions as a group and failed on both outcomes - accuracy on BIC  grounds, efficiency on convergence. The term-level diagnostics, however, showed that the `environment x reliable_node_count` interaction alone accounted for nearly all of Block H's efficiency signal (ΔAIC = +933.67 fitted individually, against the post-G model).

Block H′ tests this single term on its own. Isolating it from the other RNC interactions removes the design-matrix collinearity that prevented Block H from converging on efficiency, and applies a far smaller penalty (1 df vs 5–7) for the accuracy test.

#### What Block H′ tests

For accuracy, `C(environment):reliable_node_count_z` is already in the model via Block C. Block H′ does not apply to the accuracy model.

For efficiency, no RNC interactions survived A–G. Block H′ adds the single term:

```

In [ ]:
block_h_prime_efficiency_formula = efficiency_formula + " + C(environment):reliable_node_count_z"

In [ ]:
efficiency_h_prime = smf.negativebinomial(block_h_prime_efficiency_formula, data=df_efficiency).fit(disp=False)

**Comparison**

In [ ]:
print('BLOCK H: reliable_node_count interactions\n')
print()
keep_h_eff = block_test(efficiency_current, efficiency_h_prime, 'efficiency')
# Retain the accepted H-prime model for all final summaries and exports.
efficiency_current = efficiency_h_prime


### Summary

| Block | Terms | Accuracy | Efficiency |
|-------|-------|----------|------------|
| A      | environment × strategy                                     | drop      | **keep** |
| B      | strategy × network_kind                                    | drop      | drop |
| C      | environment × {density, clustering, path, reliable_count}  | **keep**  | drop (no convergence) |
| D      | network_kind × {density, clustering, path, reliable_count} | drop      | drop (no convergence) |
| E      | density × clustering, density × path, path × clustering    | **keep**  | **keep** |
| F      | reliable_count × {density, clustering, path}               | drop      | drop (no convergence) |
| G      | strategy × {density, clustering, path, reliable_count}     | **keep**  | drop (no convergence) |
| H      | remaining reliable_count interactions                      | drop      | drop (no convergence) |
| H′     | environment × reliable_count                               | n/a       | **keep** |

## Main Regressions

### Accuracy Model - logistic regression

In [ ]:
accuracy_final = accuracy_current
print(accuracy_final.summary())

### Efficiency model - negative binomial regression

In [ ]:
efficiency_final = efficiency_current
print(efficiency_final.summary())

## Diagnostic Checks

**Accuracy**

In [ ]:
print('FINAL ACCURACY MODEL\n')
print(f'Formula:{accuracy_final.model.formula}')
print()
print(f'Converged: {accuracy_final.mle_retvals.get("converged", "n/a")}')
print(f'AIC: {accuracy_final.aic:,.1f}')
print(f'BIC: {accuracy_final.bic:,.1f}')
print(f'Pseudo R²: {accuracy_final.prsquared:.4f}')
print(f'Log-likelihood: {accuracy_final.llf:,.1f}')
print(f'Largest |coef|: {accuracy_final.params.abs().max():.3f}')
print(f'Largest SE: {accuracy_final.bse.max():.3f}')
print()

**Efficiency**

In [ ]:
print('FINAL EFFICIENCY MODEL\n')
print(f'Formula: {efficiency_final.model.formula}')
print()
print(f'Converged: {efficiency_final.mle_retvals.get("converged", "n/a")}')
print(f'Iterations: {efficiency_final.mle_retvals.get("iterations", "n/a")}')
print(f'AIC: {efficiency_final.aic:,.1f}')
print(f'BIC: {efficiency_final.bic:,.1f}')
print(f'Pseudo R²: {efficiency_final.prsquared:.4f}')
print(f'Log-likelihood: {efficiency_final.llf:,.1f}')
print(f'Largest |coef|: {efficiency_final.params.abs().max():.3f}')
print(f'Largest SE: {efficiency_final.bse.max():.3f}')
print(f'Estimated alpha: {efficiency_final.params["alpha"]:.3f}')
print()

# Results

## Main Results

### Coefficient Table

In [ ]:
def coef_table(model, model_name):
    """
    This function returns the structured coeffiecient table with all requiered values
    """
    params = model.params
    se = model.bse
    p_values = model.pvalues
    confidence_intervals = model.conf_int()
    keep = [p for p in params.index if p != 'alpha']
    
    table = pd.DataFrame({
        'coef': params[keep].round(3),
        '95% CI low': confidence_intervals.loc[keep, 0].round(3),
        '95% CI high': confidence_intervals.loc[keep, 1].round(3),
        'p-value': p_values[keep].apply(lambda p: f'{p:.3g}')})
    
    print(f'{model_name}')
    print(f'AIC: {model.aic:,.1f}')
    print(f'BIC: {model.bic:,.1f}')
    print(f'Pseudo R^2: {model.prsquared:.4f}')
    print(f'N obs: {int(model.nobs):,}')
    print()
    return table

In [ ]:
accuracy_table = coef_table(accuracy_current, 'FINAL ACCURACY MODEL')
accuracy_table.to_csv('results/main-analysis/accuracy_model_results.csv')
accuracy_table

In [ ]:
efficiency_table = coef_table(efficiency_current, 'FINAL EFFICIENCY MODEL')
efficiency_table.to_csv('results/main-analysis/efficiency_model_results.csv')
efficiency_table

# Robustness Analysis

This final section re-runs the block-selection procedure under alternative block orders (Step 13) and selection rules (Step 14), then checks prediction-level stability (Step 15), summarises (Step 16) and interprets (Step 17). It of course uses the same core dataset as the main analysis (`df_accuracy`, `df_efficiency`).

Block term strings are identical to the main analysis above. Block H differs by outcome (accuracy already has environment x RNC and strategy x RNC via Blocks C and G, so it adds the remaining four, efficiency adds all six). Block H′ (environment x RNC alone) is efficiency-only and is None for accuracy.

The robustness checks re-run the blockwise selection of the main analysis under different block orders (Step 13) and a more permissive selection rule (Step 14), then verify the substantive conclusions at the level of predicted outcomes (Step 15) before summarising (Steps 16–17). They use the same `df_accuracy` and `df_efficiency` built above.

The machinery is small: each interaction block is a right-hand-side fragment, and a single `run_selection` replays the iterative addition for a given outcome, order, and rule. Block **H** collects the remaining reliable-count interactions (it differs by outcome, since the accuracy model already absorbs `environment × RNC` and `strategy × RNC` through blocks C and G), and block **H′** is the single `environment × reliable_node_count` term, which applies to efficiency only.

In [ ]:
STRUCT = ['density_z', 'clustering_z', 'path_length_z', 'reliable_node_count_z']

In [ ]:
def _b(*terms):
    return ' + '.join(terms)

In [ ]:
# Candidate interaction blocks as RHS fragments. A dict value means the block differs by outcome also H′ is None for accuracy because that term is already in block C.
BLOCKS = {
    'A':  _b('C(environment):C(strategy)'),
    'B':  _b('C(strategy):C(network_kind)'),
    'C':  _b(*[f'C(environment):{s}' for s in STRUCT]),
    'D':  _b(*[f'C(network_kind):{s}' for s in STRUCT]),
    'E':  _b('density_z:clustering_z', 'density_z:path_length_z', 'path_length_z:clustering_z'),
    'F':  _b(*[f'reliable_node_count_z:{s}' for s in STRUCT if s != 'reliable_node_count_z']),
    'G':  _b(*[f'C(strategy):{s}' for s in STRUCT]),
    'H':  {'accuracy':   _b('C(network_kind):reliable_node_count_z',
                            *[f'reliable_node_count_z:{s}' for s in STRUCT if s != 'reliable_node_count_z']),
           'efficiency': _b(*[f'C({c}):reliable_node_count_z' for c in ['environment', 'strategy', 'network_kind']],
                            *[f'reliable_node_count_z:{s}' for s in STRUCT if s != 'reliable_node_count_z'])},
    'H′': {'accuracy': None, 'efficiency': _b('C(environment):reliable_node_count_z')},
}

In [ ]:
ORDERS = {
    'Order 1 (current)':            ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'H′'],
    'Order 2 (structure-first)':    ['E', 'A', 'B', 'C', 'D', 'F', 'G', 'H', 'H′'],
    'Order 3 (env/reliable-count)': ['A', 'H', 'H′', 'C', 'E', 'B', 'D', 'F', 'G'],
}

In [ ]:
# final retained blocks of the main analysis, for the summary table
MAIN_RETAINED = {'accuracy': {'C', 'E', 'G'}, 'efficiency': {'A', 'E', 'H′'}}

In [ ]:
def block_terms(block, kind):
    """RHS fragment for a block under one outcome, or None if it does not apply."""
    spec = BLOCKS[block]
    return spec.get(kind) if isinstance(spec, dict) else spec

In [ ]:
def fit(kind, rhs, data):
    """Fit the outcome model for a right-hand side, passing the response as an array."""
    y = data['outcome' if kind == 'accuracy' else 'steps_to_B'].to_numpy()
    X = dmatrix(rhs, data, return_type='dataframe')
    return (sm.Logit if kind == 'accuracy' else sm.NegativeBinomial)(y, X).fit(disp=False)

In [ ]:
BASE_RHS = ('C(environment) + C(strategy) + C(network_kind) + '
            'density_z + clustering_z + path_length_z + reliable_node_count_z')

In [ ]:
def run_selection(kind, order, rule='strict'):
    """
    Replay the blockwise selection for one outcome and block order.
    rule='strict' keeps a block on LR p<0.05, ΔAIC≥2, ΔBIC>-2 and convergence;
    rule='aic' drops the BIC requirement. Returns (final_model, {block: status}).
    """
    data = df_accuracy if kind == 'accuracy' else df_efficiency
    rhs, current, status = BASE_RHS, fit(kind, BASE_RHS, data), {}

    for b in order:
        terms = block_terms(b, kind)
        if terms is None:
            status[b] = 'n/a'
            continue
        new = fit(kind, rhs + ' + ' + terms, data)
        improves = (chi2.sf(2 * (new.llf - current.llf), new.df_model - current.df_model) < 0.05
                    and current.aic - new.aic >= 2
                    and (rule == 'aic' or current.bic - new.bic > -2))
        if not new.mle_retvals.get('converged', True):
            status[b] = 'failed'
        elif improves:
            status[b], rhs, current = 'kept', rhs + ' + ' + terms, new
        else:
            status[b] = 'dropped'

    return current, status

## Step 13) Block-order sensitivity

Re-run the selection under all three orders for each outcome and report which blocks are kept, dropped, or non-convergent, the final AIC/BIC, and any coefficient sign changes versus the current order. Sign flips are expected in interaction-heavy models and are cross-checked against the predictions in Step 15.

In [ ]:
order_models = {kind: {name: run_selection(kind, order) for name, order in ORDERS.items()}
                for kind in ('accuracy', 'efficiency')}

In [ ]:
for kind in ('accuracy', 'efficiency'):
    print(f'- {kind.upper()} -')
    for name, (model, status) in order_models[kind].items():
        kept = [b for b in ORDERS[name] if status[b] == 'kept']
        print(f'{name}: kept {kept}   AIC {model.aic:,.1f}   BIC {model.bic:,.1f}')
        for tag in ('failed', 'dropped'):
            blocks = [b for b in ORDERS[name] if status[b] == tag]
            if blocks:
                print(f'    {tag}: {blocks}')
    print()

In [ ]:
for kind in ('accuracy', 'efficiency'):
    ref = order_models[kind]['Order 1 (current)'][0].params
    print(f'{kind}: coefficient sign changes vs Order 1')
    for name in ('Order 2 (structure-first)', 'Order 3 (env/reliable-count)'):
        other = order_models[kind][name][0].params
        flips = [k for k in ref.index
                 if k in other.index and k != 'alpha' and np.sign(ref[k]) != np.sign(other[k])]
        print(f'  {name}: {flips or "none"}')
    print()

## Step 14) Selection-rule sensitivity

Compare the strict rule (LR p<0.05, ΔAIC≥2, ΔBIC>-2) against an AIC-sensitive rule that drops the BIC requirement, both on the current order. Blocks that change status are kept by AIC but trimmed by the BIC penalty.

In [ ]:
for kind in ('accuracy', 'efficiency'):
    print(f'- {kind.upper()} (Order 1) -')
    kept = {}
    for rule in ('strict', 'aic'):
        model, status = run_selection(kind, ORDERS['Order 1 (current)'], rule)
        kept[rule] = [b for b in status if status[b] == 'kept']
        print(f'  {rule:6s}: kept {kept[rule]}   AIC {model.aic:,.1f}   BIC {model.bic:,.1f}')
    print(f'  changes status: {sorted(set(kept["strict"]) ^ set(kept["aic"])) or "(none)"}')
    print()

## Step 15) Prediction-level robustness

Check the substantive conclusions at the level of predicted outcomes, which are stable even where coefficient signs are not. Report overall and by-condition predictions, plus predicted outcomes at low / medium / high of each structural predictor (others held at the mean), by environment.

In [ ]:
df_accuracy['predicted_success_prob'] = accuracy_final.predict(df_accuracy)
df_efficiency['predicted_steps_to_B'] = efficiency_final.predict(df_efficiency)

In [ ]:
print('Overall predicted success rate:', round(df_accuracy['predicted_success_prob'].mean(), 4))
print(df_accuracy.groupby(['environment', 'strategy'])['predicted_success_prob'].mean().round(4).to_string())
print(df_accuracy.groupby('network_kind')['predicted_success_prob'].mean().round(4).to_string())
print()
print('Overall predicted steps to B:', round(df_efficiency['predicted_steps_to_B'].mean(), 1))
print(df_efficiency.groupby(['environment', 'strategy'])['predicted_steps_to_B'].mean().round(1).to_string())
print(df_efficiency.groupby('network_kind')['predicted_steps_to_B'].mean().round(1).to_string())

In [ ]:
def predicted_by_structure(model, data, var):
    """Mean prediction at var = -1, 0, +1 (other structural predictors at 0), by environment."""
    rows = []
    for z in (-1.0, 0.0, 1.0):
        grid = data.assign(**{s: 0.0 for s in STRUCT})
        grid[var] = z
        rows.append({'z': z, **{env: model.predict(grid.assign(environment=env)).mean()
                                for env in ('disinfo', 'misinfo')}})
    return pd.DataFrame(rows).set_index('z')

In [ ]:
for var in STRUCT:
    print(f'ACCURACY — P(success) across {var}, by environment')
    print(predicted_by_structure(accuracy_final, df_accuracy, var).round(4).to_string(), '\n')

In [ ]:
for var in ['density_z', 'clustering_z', 'path_length_z']:
    print(f'EFFICIENCY — steps to B across {var}, by environment')
    print(predicted_by_structure(efficiency_final, df_efficiency, var).round(1).to_string(), '\n')

## Step 16) Robustness summary table

Each effect is classified **Robust** (kept in every applicable order under the strict rule), **Partly robust** (kept in some, or only under the AIC-sensitive rule), **Fragile** (never kept and not merely a BIC casualty), or **Not stably estimable** (non-convergence). First-order effects are always present, so their stability is read from the Step 15 predictions.

In [ ]:
LABELS = [('environment × strategy', 'A'), ('environment × structure', 'C'),
          ('strategy × structure', 'G'),   ('network_kind × structure', 'D'),
          ('structure × structure', 'E'),  ('environment × reliable count', 'H′'),
          ('strategy × reliable count', 'G'), ('reliable count × structure', 'F')]

FIRST_ORDER = ['environment main effect', 'strategy main effect', 'network kind',
               'density', 'clustering', 'path length', 'reliable node count']

In [ ]:
def classify(kind, block, aic_kept):
    """Return (alt-orders string, AIC-rule status, verdict) for one block."""
    applicable = [n for n in ORDERS if order_models[kind][n][1].get(block) != 'n/a']
    if not applicable:
        return 'n/a', 'n/a', 'n/a for this outcome'
    n_kept = sum(order_models[kind][n][1][block] == 'kept' for n in applicable)
    failed = any(order_models[kind][n][1][block] == 'failed' for n in applicable)
    in_aic = block in aic_kept
    if n_kept == len(applicable):
        verdict = 'Robust'
    elif n_kept > 0 or in_aic:
        verdict = 'Partly robust'
    elif failed:
        verdict = 'Not stably estimable'
    else:
        verdict = 'Fragile'
    return f'{n_kept}/{len(applicable)} orders', ('retained' if in_aic else 'dropped'), verdict

In [ ]:
def summarise(kind):
    aic_status = run_selection(kind, ORDERS['Order 1 (current)'], 'aic')[1]
    aic_kept = {b for b, s in aic_status.items() if s == 'kept'}

    rows = [{'Effect / block': e, 'Main model': 'always present',
             'Alt. orders': 'always present', 'AIC rule': 'always present',
             'Verdict': 'see Step 15'} for e in FIRST_ORDER]

    for label, block in LABELS:
        alt, aic, verdict = classify(kind, block, aic_kept)
        main = 'n/a' if alt == 'n/a' else ('retained' if block in MAIN_RETAINED[kind] else 'dropped')
        rows.append({'Effect / block': label, 'Main model': main,
                     'Alt. orders': alt, 'AIC rule': aic, 'Verdict': verdict})
    return pd.DataFrame(rows)

In [ ]:
acc_summary = summarise('accuracy')
eff_summary = summarise('efficiency')
print('ACCURACY\n', acc_summary.to_string(index=False), '\n')
print('EFFICIENCY\n', eff_summary.to_string(index=False))

## Step 17) Interpretation of robustness results

**Firstly we can say that Accuracy is stable.** environment x structure (C), strategy x structure (G) and structure x structure (E) are retained in all three orders with identical fit, and survive the AIC-sensitive rule. network_kind x structure (D) and reliable_count × structure (F) are partly robust: they improve AIC but are trimmed by BIC, so they belong in supporting analysis rather than headline claims. Finally environment × strategy is fragile.

**As for Efficiency we can say that it is substantively stable but fragile in block membership.** environment x strategy (A) is robust. structure x structure (E) and environment x reliable_count (H′) are retained in the main order, their drop in some alternative orders is path-dependence (in this case an earlier block absorbs the same variance), and the Step 15 predictions confirm the underlying patterns hold, so they are reported as partly robust. The blocks that fail to converge (environment x structure, network_kind x structure, and the grouped reliable-count block) are recorded as not stably estimable - a limitation of the negative-binomial fit under this design, not evidence that the interactions are absent.

The single coefficient sign change (path_length_z under Order 3) is not substantive: predicted steps fall monotonically with path length in both environments across every specification, so the predicted pattern, not the sign, is the finding.